In [3]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image
from sklearn.decomposition import PCA
from transformers import AutoImageProcessor, AutoModel

# 1. Load model and processor
model_id = "facebook/dinov3-vitl16-pretrain-lvd1689m"
processor = AutoImageProcessor.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)
model.eval()

# 2. Prepare image & forward pass
image = Image.open("your_image.jpg").convert("RGB")
inputs = processor(images=image, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

# Shape: [1, sequence_length, hidden_dim]
tokens = outputs.last_hidden_state

# 3. Determine spatial grid dimensions
# ViT-L/16 patch size is 16x16
patch_size = 16
h_patches = inputs["pixel_values"].shape[-2] // patch_size
w_patches = inputs["pixel_values"].shape[-1] // patch_size
num_patches = h_patches * w_patches

# 4. Extract ONLY spatial patch tokens
# ViT typically includes a [CLS] token and optionally register tokens at the beginning.
# Slice the last (h_patches * w_patches) tokens to get the spatial grid.
patch_tokens = tokens[0, -num_patches:, :].cpu().numpy()  # [num_patches, 1024]

# 5. Apply PCA to reduce 1024-d -> 3-d (RGB)
pca = PCA(n_components=3)
pca_features = pca.fit_transform(patch_tokens)  # [num_patches, 3]

# 6. Normalize per channel to [0, 1] range
pca_min = pca_features.min(axis=0, keepdims=True)
pca_max = pca_features.max(axis=0, keepdims=True)
pca_features = (pca_features - pca_min) / (pca_max - pca_min + 1e-8)

# 7. Reshape into an image grid [h_patches, w_patches, 3]
pca_map = pca_features.reshape(h_patches, w_patches, 3)

# 8. Plot side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(image)
axes[0].set_title("Original Image")
axes[0].axis("off")

# Display PCA heatmap resized to match original aspect ratio
axes[1].imshow(pca_map, interpolation="nearest")
axes[1].set_title("DINO Patch Features (PCA 1-3 as RGB)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m.
401 Client Error. (Request ID: Root=1-6a9499cd-2e012b8f025ad595060491e5;edaa4ec6-4dd6-4e98-b792-ef03a5ca877d)

Cannot access gated repo for url https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m/resolve/main/processor_config.json.
Access to model facebook/dinov3-vitl16-pretrain-lvd1689m is restricted. You must have access to it and be authenticated to access it. Please log in.